# Telco Customer Churn Prediction

**Goal:** predict which telecom customers are likely to leave (churn), and find out what drives it.

**Pipeline:** load data → clean → split → train models → check results → business findings

Data: Kaggle "Telco Customer Churn" (`WA_Fn-UseC_-Telco-Customer-Churn.csv`). Run the cells from top to bottom.

## Step 1: Load the data
Run the next cell and choose the CSV file when the upload window opens. (Colab deletes uploaded files when a session resets, so do this first each time.)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

## Step 2: Explore the data
How many rows and columns, what types, and how many customers churn?

In [ ]:
df.info()

In [ ]:
df['Churn'].value_counts()

About 27% of customers churn (1,869 of 7,043). The data is imbalanced, so accuracy alone is misleading: a model that always says "stays" would be about 73% accurate and useless.

## Step 3: Clean the data
`TotalCharges` is stored as text because 11 new customers (tenure = 0) have a blank value. They have not been billed yet, so we fill the blanks with 0 and convert the column to numbers.

In [ ]:
df[df['TotalCharges'] == ' ']

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df['TotalCharges'].dtype

Convert the target column to numbers: Yes = 1 (churned), No = 0 (stayed). Then turn the other text columns into True/False columns (one-hot encoding), because models need numbers.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df_encoded = pd.get_dummies(df, columns=['gender', 'Partner', 'Dependents', 'PhoneService',
                                         'MultipleLines', 'InternetService', 'OnlineSecurity',
                                         'OnlineBackup', 'DeviceProtection', 'TechSupport',
                                         'StreamingTV', 'StreamingMovies', 'Contract',
                                         'PaperlessBilling', 'PaymentMethod'])

# customerID is just a label, it does not help predict churn
df_encoded = df_encoded.drop('customerID', axis=1)
df_encoded.head()

## Step 4: Split into clues (X), answer (y), and train/test sets
80% of customers teach the model, 20% are held back as an exam it has not seen.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('Churn', axis=1)   # clues
y = df_encoded['Churn']                # the answer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

## Step 5: Model 1, Logistic Regression (baseline)
Features are scaled first, because columns like `TotalCharges` (thousands) and the True/False columns are on very different scales.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, predictions))
print()
print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Reading the results for churners (row 1): **recall** is how many of the real churners the model finds, and **precision** is how many of the customers it flags really churn. The baseline finds about 60% of churners.

## Step 6: Model 2, Random Forest
Many decision trees that vote. Trees do not need scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, rf_predictions))
print()
print(confusion_matrix(y_test, rf_predictions))
print()
print(classification_report(y_test, rf_predictions))

The Random Forest did **worse** than the simple baseline: it finds only about 46% of churners. A more complex model is not automatically better.

## Step 7: Model 3, balanced Logistic Regression
`class_weight='balanced'` makes a mistake on a churner count for more, so the model pays more attention to them.

In [ ]:
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_balanced.fit(X_train_scaled, y_train)
lr_predictions = lr_balanced.predict(X_test_scaled)

print(confusion_matrix(y_test, lr_predictions))
print()
print(classification_report(y_test, lr_predictions))

Recall for churners rises to about 0.82 (it finds 82 of every 100 real churners). The price is lower precision (about 0.51), so more false alarms. Which model is best is a business decision: it depends on what a missed churner costs compared with an unnecessary retention offer.

## Step 8: What drives churn?
Positive numbers push toward churn, negative numbers push toward staying (features were scaled, so sizes can be compared).

In [ ]:
importance = pd.Series(lr_balanced.coef_[0], index=X.columns)
print(importance.sort_values(ascending=False).head(8))
print()
print(importance.sort_values().head(8))

Check the main patterns directly in the data. `tenure`, `MonthlyCharges` and `TotalCharges` overlap, so do not over-read their individual signs.

In [ ]:
print(df.groupby('Contract')['Churn'].mean())
print()
print(df.groupby('InternetService')['Churn'].mean())

## Step 9: Business findings and impact

- Customers on **month-to-month** contracts churn at about 42.7%, against 2.8% for two-year contracts.
- **Fiber optic** customers churn at about 41.9%, against 19.0% for DSL and 7.4% for no internet.
- Low **tenure** (new customers) is the strongest churn signal.

**Example impact (made-up numbers to show scale):** if a plan saves 10% of the 1,869 churners, that is about 187 customers. At an example value of 65 per month (780 per year), that protects about 145,860 in yearly income.

In [ ]:
churners = 1869
saved = churners * 0.10
yearly_value = 65 * 12
print("Customers kept:", round(saved))
print("Yearly income protected:", round(saved) * yearly_value)